In [345]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import json
import os
from scipy.stats import pearsonr, spearmanr
from sklearn.metrics import classification_report
from scipy import stats
import warnings

In [2]:
from sqlalchemy import create_engine, insert, text, inspect,Table, Column,Integer, String, DateTime, Float, MetaData
from datetime import datetime
from sqlalchemy.orm import sessionmaker
from sqlalchemy.orm import declarative_base
base=declarative_base

## get data

In [3]:
def create_engine_session(connection_string):
    try:
        engine = create_engine(
            connection_string, 
            fast_executemany=True,
            pool_pre_ping=True,
            pool_recycle=3600,  # 1 hour
            pool_size=10,
            max_overflow=20
            )

        # test engine
        with engine.connect() as connection:
            connection.execute(text("select 1"))
            print("Successfully created and tested SQLAlchemy engine.")
        my_Session = sessionmaker(bind=engine)

    except Exception as db_err:
        print("Failed to create SQLAlchemy engine. Retrying attempt  DB error: %s", db_err)
    return engine, my_Session

In [4]:
def get_data_from_sql(engine, query):

    with engine.connect() as connection:
        result = pd.read_sql(query, connection)
    
    return result

In [15]:
db_type_name_dev='dev'
db_name_dev='SimulMl'
connection_string_dev=f"mssql+pyodbc://sql{db_type_name_dev}2016/{db_name_dev}?driver=ODBC Driver 18 for SQL Server&TrustServerCertificate=yes"
temp_text='_temp'
print(connection_string_dev)

mssql+pyodbc://sqldev2016/SimulMl?driver=ODBC Driver 18 for SQL Server&TrustServerCertificate=yes


In [10]:
engine_dev, my_Session_dev=create_engine_session(connection_string_dev)

Successfully created and tested SQLAlchemy engine.


### input data

In [11]:
input_query="""
select *  from
sqldev2016.SimulMl.dbo.ML_InputSimul; 
"""


In [12]:
df_input= get_data_from_sql(engine_dev, input_query)

In [19]:
df_input.shape

(44472, 43)

In [13]:
df_input.head()

,Id_Row,mezaheReshumaRatz,KodSeker,TypeSeker,TypeSeker_Descr,ShnatSeker,ChodeshSeker,ShemAvoda,SugAvoda,ShemMachlaka,...,SemelRakaz,MenaheletMi,MenaheletMi_Descr,SemelAnafFromSeker,SemelAnafFromSeker_Descr,SentDate,RetrieveDate,Tz,TzSentDate,TZRetrieveDate
0,269991,112512123485341,12,3,סימול משלח ללא ענף ידוע,2025,12,,,,...,45.0,NaN,None,None,None,2025-12-24 06:35:03.070,2026-07-06 12:50:08.627,None,None,None
1,269992,112512123485941,12,3,סימול משלח ללא ענף ידוע,2025,12,,,,...,45.0,NaN,None,None,None,2025-12-24 06:35:03.070,2026-07-06 12:50:08.627,None,None,None
2,269993,112512123486611,12,3,סימול משלח ללא ענף ידוע,2025,12,,,,...,43.0,NaN,None,None,None,2025-12-24 06:35:03.070,2026-07-06 12:50:08.627,None,None,None
3,269994,112512123486751,12,3,סימול משלח ללא ענף ידוע,2025,12,,,,...,43.0,NaN,None,None,None,2025-12-24 06:35:03.070,2026-07-06 12:50:08.627,None,None,None
4,269995,112512123486761,12,3,סימול משלח ללא ענף ידוע,2025,12,,,,...,43.0,NaN,None,None,None,2025-12-24 06:35:03.070,2026-07-06 12:50:08.627,None,None,None


### results data

In [16]:
query_result= f""" 
select * from
sql{db_type_name_dev}2016.simulML.[dbo]. [ML_ResultSimul{temp_text}]
"""

In [17]:
df_result= get_data_from_sql(engine_dev, query_result)

In [18]:
df_result.shape

(44472, 16)

In [20]:
df_result.head(2)

,Id_Row,mezaheReshumaRatz,KodSeker,ProjectId,PredictedAnaf1,Score_PredictedAnaf1,PredictedMishlah1,Score_PredictedMishlah1,PredictedAnaf2,Score_PredictedAnaf2,PredictedMishlah2,Score_PredictedMishlah2,SentDate,RetrievalDate,StatusTorML,Deployed_Enviroment_Version
0,247489,126158074857211,2,1,None,NaN,2359,1.833,None,NaN,2356,1.5,2026-07-06 14:50:22.210,None,1,v0.230
1,247490,126158074857212,2,1,None,NaN,2359,2.000,None,NaN,1345,1.5,2026-07-06 14:50:22.210,None,2,v0.230


### mishlah data

In [22]:
query_mishlah= f""" 
select * from
sql{db_type_name_dev}2016.simulML.[dbo]. [ML_MishlahSimulByModels{temp_text}]
"""

In [31]:
df_mishlah= get_data_from_sql(engine_dev, query_mishlah)

In [32]:
df_mishlah.shape

(360480, 8)

In [33]:
df_mishlah.head(2)

,mezaheReshumaRatz,KodSeker,ProjectId,ModelId_PredictedMishlah,PredictedMishlah,Score_PredictedMishlah,CreationDate,Deployed_Enviroment_Version
0,112510123177951,12,1,bert_mishlah_v5,"[""8344"", ""8332"", ""7511""]","[0.8876, 0.9745, 0.8919]",2026-06-03 13:51:46.217,v0.230
1,112510123177951,12,1,embedding_mishlah_v2,"[""8344"", ""0"", ""0""]","[0.9535, 0.0, 0.0]",2026-06-03 13:51:46.217,v0.230


### anaf data

In [27]:
query_anaf= f""" 
select * from
sql{db_type_name_dev}2016.simulML.[dbo]. [ML_AnafSimulByModels{temp_text}]
"""

In [28]:
df_anaf= get_data_from_sql(engine_dev, query_anaf)

In [29]:
df_anaf.shape

(93120, 8)

In [30]:
df_anaf.head()

,mezaheReshumaRatz,KodSeker,ProjectId,ModelId_PredictedAnaf,PredictedAnaf,Score_PredictedAnaf,CreationDate,Deployed_Enviroment_Version
0,11511123508531,1,1,bert_anaf_v5,"[""85XX"", ""8XXX"", ""8525""]","[0.8644, 0.2595, 0.5088]",2026-07-06 18:08:03.090,v0.234
1,11511123508531,1,1,embedding_anaf_v2,"[""8513"", ""85XX"", ""0""]","[0.6909, 0.6225, 0.0]",2026-07-06 18:08:03.090,v0.234
2,11511123508531,1,1,ReRanker_Anaf_v3,"[""85XX"", ""8580"", ""8513""]","[0.6957605985037406, 0.6583333333333333, 0.762...",2026-07-06 18:08:03.090,v0.234
3,11511123508531,1,1,xgb_anaf_e5_v4,"[""85XX"", ""8720"", ""XXXX""]","[0.6593, 0.5955, 0.6805]",2026-07-06 18:08:03.090,v0.234
4,11511123508541,1,1,bert_anaf_v5,"[""8560"", ""8524"", ""8523""]","[0.8034, 0.7835, 0.8141]",2026-07-06 18:08:03.090,v0.234


### test_2025

In [238]:
test_path_csv="/home/nfsdisk/Simul_AI/new_datas/processed/processed_test_data.csv"

In [239]:
df_test = pd.read_csv(test_path_csv, encoding='utf-8-sig')
df_test['mezaheReshumaRaz']=df_test['mezaheReshumaRaz'].astype(str)

In [255]:
df_test=df_test.rename(columns={'mezaheReshumaRaz':'mezaheReshumaRatz'})

In [256]:
df_test.shape

(46163, 40)

In [257]:
df_test.head()

,mezaheReshumaRatz,kodseker,ShemSeker,ShnatSeker,ChodeshSeker,HaimNivdak,YeshuvAvoda,YeshuvAvoda_descr,MaamadAvoda,MaamadAvoda_descr,...,SamlanUpdate,gil,MenahelEtMi,MenaheletMi_descr,StatusIhutSimul,Dira,Prat,tz,TypeSeker,TypeSeker_descr
0,112510123177951,12,"סכ""א מקצוע",2025.0,10.0,0,NaN,NaN,NaN,NaN,...,0.0,54.0,NaN,NaN,5,2317795,1,28178663.0,3,סימול משלח ללא ענף ידוע
1,112510123188591,12,"סכ""א מקצוע",2025.0,10.0,0,NaN,NaN,NaN,NaN,...,0.0,54.0,NaN,NaN,2,2318859,1,28619534.0,3,סימול משלח ללא ענף ידוע
2,112510123189301,12,"סכ""א מקצוע",2025.0,10.0,0,NaN,NaN,NaN,NaN,...,0.0,26.0,NaN,NaN,2,2318930,1,207919127.0,3,סימול משלח ללא ענף ידוע
3,112510123190931,12,"סכ""א מקצוע",2025.0,10.0,0,NaN,NaN,NaN,NaN,...,0.0,62.0,NaN,NaN,5,2319093,1,58376690.0,3,סימול משלח ללא ענף ידוע
4,112510123191211,12,"סכ""א מקצוע",2025.0,10.0,0,NaN,NaN,NaN,NaN,...,0.0,32.0,NaN,NaN,2,2319121,1,204542849.0,3,סימול משלח ללא ענף ידוע


In [296]:
query_test= "select * from sqldev2016.SimulML.[dbo].[TargetDataAmir]"

In [297]:
df_test= get_data_from_sql(engine_dev, query_test)

In [299]:
df_test.head(2)

,mezaheReshumaRatz,SemelAnafSofi,SemelMishlahSofi,kodseker,Teur
0,112512123485341,None,2529,12,סימול משלח ללא ענף ידוע
1,112512123485941,None,5322,12,סימול משלח ללא ענף ידוע


In [300]:
df_test.shape

(44302, 5)

## tests

### test results

In [46]:
df_result.head(2).T

,0,1
Id_Row,247489,247490
mezaheReshumaRatz,126158074857211,126158074857212
KodSeker,2,2
ProjectId,1,1
PredictedAnaf1,None,None
Score_PredictedAnaf1,NaN,NaN
PredictedMishlah1,2359,2359
Score_PredictedMishlah1,1.833,2.0
PredictedAnaf2,None,None
Score_PredictedAnaf2,NaN,NaN


In [35]:
if len(df_result)!= len(df_input):
    raise RuntimeError("results and input are not the same number of rows")

#### check number of rows per types 

In [41]:
df_input.groupby('RetrieveDate')['mezaheReshumaRatz'].count().reset_index()

,RetrieveDate,mezaheReshumaRatz
0,2026-07-06 12:50:06.933,23280
1,2026-07-06 12:50:08.627,21192


In [71]:
df_input.groupby(['TypeSeker','KodSeker'])['mezaheReshumaRatz'].count().reset_index()

,TypeSeker,KodSeker,mezaheReshumaRatz
0,1,1,16235
1,1,3,2435
2,1,4,4610
3,2,2,19969
4,3,12,1223


In [39]:
df_result.groupby('SentDate')['mezaheReshumaRatz'].count().reset_index()

,SentDate,mezaheReshumaRatz
0,2026-07-06 14:50:22.210,21192
1,2026-07-06 18:08:03.090,23280


In [52]:
df_result.columns

Index(['Id_Row', 'mezaheReshumaRatz', 'KodSeker', 'ProjectId',
       'PredictedAnaf1', 'Score_PredictedAnaf1', 'PredictedMishlah1',
       'Score_PredictedMishlah1', 'PredictedAnaf2', 'Score_PredictedAnaf2',
       'PredictedMishlah2', 'Score_PredictedMishlah2', 'SentDate',
       'RetrievalDate', 'StatusTorML', 'Deployed_Enviroment_Version'],
      dtype='object')

In [83]:
df_result.merge(df_input[['TypeSeker','KodSeker']].drop_duplicates(), on='KodSeker').groupby('TypeSeker').agg({'PredictedAnaf1':(['min',lambda x:  x.dropna().astype(str).min()],
                                                                                                                                  ['max',lambda x:  x.dropna().astype(str).max()], 'count'),
                                                                                                               'Score_PredictedAnaf1':('min', 'max', 'count'),
                                                                                                               'PredictedAnaf2':(['min',lambda x:  x.dropna().astype(str).min()],
                                                                                                                                  ['max',lambda x:  x.dropna().astype(str).max()], 'count'),
                                                                                                               'Score_PredictedAnaf2':('min', 'max', 'count'),
                                                                                                               'PredictedMishlah1':(['min',lambda x:  x.dropna().astype(str).min()], 
                                                                                                                                    ['max',lambda x:  x.dropna().astype(str).max()], 'count'),
                                                                                                               'Score_PredictedMishlah1':('min', 'max', 'count'),
                                                                                                               'PredictedMishlah2':(['min',lambda x:  x.dropna().astype(str).min()],
                                                                                                                                     ['max',lambda x:  x.dropna().astype(str).max()], 'count'),
                                                                                                               'Score_PredictedMishlah2':('min', 'max', 'count')}).T

TypeSeker                          1      2      3
PredictedAnaf1          min     0113    NaN    NaN
                        max     XXXX    NaN    NaN
                        count  23279      0      0
Score_PredictedAnaf1    min      0.0    NaN    NaN
                        max      4.0    NaN    NaN
                        count  23280      0      0
PredictedAnaf2          min     0113    NaN    NaN
                        max     XXXX    NaN    NaN
                        count  23197      0      0
Score_PredictedAnaf2    min      0.0    NaN    NaN
                        max      3.0    NaN    NaN
                        count  23280      0      0
PredictedMishlah1       min     1111   1114   1120
                        max     XXXX   XXXX   XXXX
                        count  23280  19969   1223
Score_PredictedMishlah1 min    0.333  0.333    1.0
                        max      4.0    4.0    4.0
                        count  23280  19969   1223
PredictedMishlah2       min     1111   1114   1120
                        max     XXXX   XXXX   XXXX
                        count  23231  19946   1223
Score_PredictedMishlah2 min      0.0    0.0  0.333
                        max      3.0    3.0  2.833
                        count  23280  19969   1223

#### test if there are several versions in results

In [79]:
df_result.merge(df_input[['TypeSeker','KodSeker']].drop_duplicates(), on='KodSeker').groupby(['TypeSeker','Deployed_Enviroment_Version']).agg({'mezaheReshumaRatz':'count','PredictedMishlah1':'count', 'PredictedAnaf1':'count'})

,,mezaheReshumaRatz,PredictedMishlah1,PredictedAnaf1
TypeSeker,Deployed_Enviroment_Version,,,
1,v0.234,23280,23280,23279
2,v0.230,19969,19969,0
3,v0.230,1223,1223,0


In [88]:
if df_result['Deployed_Enviroment_Version'].nunique()>1:
    raise RuntimeError(f"there are {df_result['Deployed_Enviroment_Version'].nunique()}  versions on the same input")

RuntimeError: there are 2  versions on the same input

#### test if anf is null or mishlah is null in results table

In [105]:
df_result[(df_result['PredictedAnaf1'].isnull()) & (df_result['KodSeker'].isin([1,3,4]))]\
    .merge(df_input,on='mezaheReshumaRatz', suffixes=('', '_x'))\
        .merge(df_anaf,on='mezaheReshumaRatz', suffixes=('', '_x'))\
              [['mezaheReshumaRatz','TypeSeker', 'ShemAvoda', 'SugAvoda','Score_PredictedAnaf1','PredictedAnaf1','Score_PredictedAnaf1',
                'PredictedMishlah1','RetrieveDate','Deployed_Enviroment_Version',
                'ModelId_PredictedAnaf','PredictedAnaf', 'Score_PredictedAnaf', 'CreationDate','Deployed_Enviroment_Version_x']].T

,0,1,2,3
mezaheReshumaRatz,1163123972881,1163123972881,1163123972881,1163123972881
TypeSeker,1,1,1,1
ShemAvoda,חברה למסחר,חברה למסחר,חברה למסחר,חברה למסחר
SugAvoda,חברה למסחר - מכירות מסחר כללי - סיטונאי,חברה למסחר - מכירות מסחר כללי - סיטונאי,חברה למסחר - מכירות מסחר כללי - סיטונאי,חברה למסחר - מכירות מסחר כללי - סיטונאי
Score_PredictedAnaf1,0.0,0.0,0.0,0.0
PredictedAnaf1,None,None,None,None
Score_PredictedAnaf1,0.0,0.0,0.0,0.0
PredictedMishlah1,3322,3322,3322,3322
RetrieveDate,2026-07-06 12:50:06.933000,2026-07-06 12:50:06.933000,2026-07-06 12:50:06.933000,2026-07-06 12:50:06.933000
Deployed_Enviroment_Version,v0.234,v0.234,v0.234,v0.234


In [114]:
anaf_or_mishlah_null=df_result[((df_result['PredictedAnaf1'].isnull()) & (df_result['KodSeker'].isin([1,3,4]))) |
          (df_result['PredictedMishlah1'].isnull())]
if len(anaf_or_mishlah_null)>0:
    raise RuntimeError(f" there are {len(anaf_or_mishlah_null)} rows with null in anaf or mishlah")

RuntimeError:  there are 1 rows with null in anaf or mishlah

### test anaf or mishlah

In [116]:
df_anaf.head()

,mezaheReshumaRatz,KodSeker,ProjectId,ModelId_PredictedAnaf,PredictedAnaf,Score_PredictedAnaf,CreationDate,Deployed_Enviroment_Version
0,11511123508531,1,1,bert_anaf_v5,"[""85XX"", ""8XXX"", ""8525""]","[0.8644, 0.2595, 0.5088]",2026-07-06 18:08:03.090,v0.234
1,11511123508531,1,1,embedding_anaf_v2,"[""8513"", ""85XX"", ""0""]","[0.6909, 0.6225, 0.0]",2026-07-06 18:08:03.090,v0.234
2,11511123508531,1,1,ReRanker_Anaf_v3,"[""85XX"", ""8580"", ""8513""]","[0.6957605985037406, 0.6583333333333333, 0.762...",2026-07-06 18:08:03.090,v0.234
3,11511123508531,1,1,xgb_anaf_e5_v4,"[""85XX"", ""8720"", ""XXXX""]","[0.6593, 0.5955, 0.6805]",2026-07-06 18:08:03.090,v0.234
4,11511123508541,1,1,bert_anaf_v5,"[""8560"", ""8524"", ""8523""]","[0.8034, 0.7835, 0.8141]",2026-07-06 18:08:03.090,v0.234


In [122]:
def safe_to_list(x):
    if isinstance(x, list):
        return x
    
    if pd.isna(x):
        return x
    
    if isinstance(x,str):
        return json.loads(x)
    
    return x

In [226]:
def explode_topk_df(df, target_name):
    df_1=df.copy()
    df_1[f"Score_Predicted{target_name}"]=df_1[f"Score_Predicted{target_name}"].apply(lambda x: json.loads(x))
    df_1[f'Predicted{target_name}']=df_1[f'Predicted{target_name}'].apply(safe_to_list)
    
    # explode 
    list_cols_explode=[f"Predicted{target_name}", f"Score_Predicted{target_name}"]
    df_1=df_1.explode(list_cols_explode, ignore_index=True)
    df_1["rank_predict"]=df_1.groupby(["mezaheReshumaRatz",f"ModelId_Predicted{target_name}"]).cumcount()+1

    # add column test type of prediction
    df_1['Predicted_type']=np.where(df_1[f"Predicted{target_name}"].isnull(),'null type',
                                    np.where(df_1[f"Predicted{target_name}"]=='0', 'zero',
                                             np.where(df_1[f"Predicted{target_name}"].astype("string").str.fullmatch(r"[A-Za-z0-9]{4}", na=False),'valid predict',
                                                      np.where(df_1[f"Predicted{target_name}"].astype('string').str.len().eq(3).fillna(False), 'len code 3 ',              
                                             'other, need to check'))))

    print(f"shape of original df is {df.shape} and explode df is {df_1.shape} the multification is {len(df_1)/len(df)}")

    return df_1

In [227]:
df_anaf_exp=explode_topk_df(df_anaf, 'Anaf')

shape of original df is (93120, 8) and explode df is (279360, 10) the multification is 3.0


In [228]:
df_mishlah_exp=explode_topk_df(df_mishlah, 'Mishlah')

shape of original df is (360480, 8) and explode df is (1081440, 10) the multification is 3.0


In [229]:
df_anaf.head(6)

,mezaheReshumaRatz,KodSeker,ProjectId,ModelId_PredictedAnaf,PredictedAnaf,Score_PredictedAnaf,CreationDate,Deployed_Enviroment_Version
0,11511123508531,1,1,bert_anaf_v5,"[""85XX"", ""8XXX"", ""8525""]","[0.8644, 0.2595, 0.5088]",2026-07-06 18:08:03.090,v0.234
1,11511123508531,1,1,embedding_anaf_v2,"[""8513"", ""85XX"", ""0""]","[0.6909, 0.6225, 0.0]",2026-07-06 18:08:03.090,v0.234
2,11511123508531,1,1,ReRanker_Anaf_v3,"[""85XX"", ""8580"", ""8513""]","[0.6957605985037406, 0.6583333333333333, 0.762...",2026-07-06 18:08:03.090,v0.234
3,11511123508531,1,1,xgb_anaf_e5_v4,"[""85XX"", ""8720"", ""XXXX""]","[0.6593, 0.5955, 0.6805]",2026-07-06 18:08:03.090,v0.234
4,11511123508541,1,1,bert_anaf_v5,"[""8560"", ""8524"", ""8523""]","[0.8034, 0.7835, 0.8141]",2026-07-06 18:08:03.090,v0.234
5,11511123508541,1,1,embedding_anaf_v2,"[""8560"", ""8524"", ""0""]","[0.7814, 0.7831, 0.0]",2026-07-06 18:08:03.090,v0.234


In [230]:
df_anaf_exp.head(6)

,mezaheReshumaRatz,KodSeker,ProjectId,ModelId_PredictedAnaf,PredictedAnaf,Score_PredictedAnaf,CreationDate,Deployed_Enviroment_Version,rank_predict,Predicted_type
0,11511123508531,1,1,bert_anaf_v5,85XX,0.8644,2026-07-06 18:08:03.090,v0.234,1,valid predict
1,11511123508531,1,1,bert_anaf_v5,8XXX,0.2595,2026-07-06 18:08:03.090,v0.234,2,valid predict
2,11511123508531,1,1,bert_anaf_v5,8525,0.5088,2026-07-06 18:08:03.090,v0.234,3,valid predict
3,11511123508531,1,1,embedding_anaf_v2,8513,0.6909,2026-07-06 18:08:03.090,v0.234,1,valid predict
4,11511123508531,1,1,embedding_anaf_v2,85XX,0.6225,2026-07-06 18:08:03.090,v0.234,2,valid predict
5,11511123508531,1,1,embedding_anaf_v2,0,0.0,2026-07-06 18:08:03.090,v0.234,3,zero


In [231]:
def group_rank_and_type(df, target_name):
    print('check number of prediction per valid type:')
    group_rank_type=df.groupby(['rank_predict','Predicted_type'])['mezaheReshumaRatz'].count().reset_index().rename(columns={'mezaheReshumaRatz':'cnt'})
    group_rank_type['ratio']=group_rank_type['cnt']/group_rank_type.groupby('rank_predict')['cnt'].transform('sum')
    display(group_rank_type)
    if 

    print('check number of prediction per model:')
    score_col=f'Score_Predicted{target_name}'
    group_rank_model=df.groupby(['rank_predict',f'ModelId_Predicted{target_name}']).agg(**{'cnt_valid': ('Predicted_type',
                                                                                                         lambda x: (x=='valid predict').sum()),
                                                                                                         'agg_score': (score_col, 'mean')}).reset_index()    
    group_rank_model['ratio_valid']=group_rank_model['cnt_valid']/df['mezaheReshumaRatz'].nunique()
    display(group_rank_model)

In [232]:
group_rank_and_type(df_anaf_exp, 'Anaf')

check number of prediction per valid type:


,rank_predict,Predicted_type,cnt,ratio
0,1,len code 3,2,0.000021
1,1,valid predict,93118,0.999979
2,2,len code 3,10,0.000107
3,2,valid predict,83187,0.893331
4,2,zero,9923,0.106561
5,3,len code 3,11,0.000118
6,3,null type,767,0.008237
7,3,valid predict,76623,0.822841
8,3,zero,15719,0.168804


check number of prediction per model:


,rank_predict,ModelId_PredictedAnaf,cnt_valid,agg_score,ratio_valid
0,1,ReRanker_Anaf_v3,23280,0.785608,1.000000
1,1,bert_anaf_v5,23280,0.78382,1.000000
2,1,embedding_anaf_v2,23278,0.774909,0.999914
3,1,xgb_anaf_e5_v4,23280,0.782427,1.000000
4,2,ReRanker_Anaf_v3,23280,0.590076,1.000000
5,2,bert_anaf_v5,23277,0.675485,0.999871
6,2,embedding_anaf_v2,13353,0.411325,0.573582
7,2,xgb_anaf_e5_v4,23277,0.666857,0.999871
8,3,ReRanker_Anaf_v3,22513,0.46679,0.967053
9,3,bert_anaf_v5,23277,0.637728,0.999871


In [267]:
df_anaf_exp[(df_anaf_exp[f"PredictedAnaf"].astype("string").str.startswith("0", na=False)) &
            (df_anaf_exp[f"PredictedAnaf"].astype("string").str.fullmatch(r"[A-Za-z0-9]{4}", na=False))]['mezaheReshumaRatz'].nunique()#[['ModelId_PredictedAnaf','PredictedAnaf']].drop_duplicates()

601

In [264]:
df_result.head(1)

,Id_Row,mezaheReshumaRatz,KodSeker,ProjectId,PredictedAnaf1,Score_PredictedAnaf1,PredictedMishlah1,Score_PredictedMishlah1,PredictedAnaf2,Score_PredictedAnaf2,PredictedMishlah2,Score_PredictedMishlah2,SentDate,RetrievalDate,StatusTorML,Deployed_Enviroment_Version
0,247489,126158074857211,2,1,None,NaN,2359,1.833,None,NaN,2356,1.5,2026-07-06 14:50:22.210,None,1,v0.230


In [281]:
df_result[(df_result[f"PredictedAnaf1"].astype("string").str.startswith("0", na=False)) &
            (df_result[f"PredictedAnaf1"].astype("string").str.fullmatch(r"[A-Za-z0-9]{4}", na=False))].head(10)#.to_excel('results_anaf_started_with_0.xlsx', index=False)

,Id_Row,mezaheReshumaRatz,KodSeker,ProjectId,PredictedAnaf1,Score_PredictedAnaf1,PredictedMishlah1,Score_PredictedMishlah1,PredictedAnaf2,Score_PredictedAnaf2,PredictedMishlah2,Score_PredictedMishlah2,SentDate,RetrievalDate,StatusTorML,Deployed_Enviroment_Version
15322,263021,11512123505151,1,1,0401,4.000,6121,4.0,0404,1.333,8160,0.333,2026-07-06 18:08:03.090,None,2,v0.234
15323,263022,11512123505161,1,1,0XXX,2.000,1311,1.0,0161,1.000,XXXX,0.333,2026-07-06 18:08:03.090,None,1,v0.234
15857,262772,11512123484041,1,1,0123,3.500,6112,4.0,012X,0.667,6113,0.667,2026-07-06 18:08:03.090,None,2,v0.234
21643,263197,11512123519341,1,1,0117,2.000,4120,4.0,0113,1.500,6111,1.000,2026-07-06 18:08:03.090,None,2,v0.234
21761,263315,11512123525081,1,1,0XXX,2.000,4110,3.5,XXXX,0.833,4XXX,1.500,2026-07-06 18:08:03.090,None,2,v0.234
21922,263476,11512223484051,1,1,0130,1.333,3313,4.0,4621,1.000,6113,0.500,2026-07-06 18:08:03.090,None,1,v0.234
22036,263590,11512223493191,1,1,0891,3.333,4321,4.0,2011,0.833,3323,1.000,2026-07-06 18:08:03.090,None,2,v0.234
22080,263634,11512223502291,1,1,0151,1.000,3313,4.0,4665,1.000,8332,0.500,2026-07-06 18:08:03.090,None,1,v0.234
22176,263730,11512223514511,1,1,0120,3.000,6112,4.0,0123,0.500,6130,0.667,2026-07-06 18:08:03.090,None,2,v0.234
22859,264413,1161123659531,1,1,0113,1.333,6112,3.5,0150,1.000,6114,1.500,2026-07-06 18:08:03.090,None,1,v0.234


In [218]:
df_anaf_exp[df_anaf_exp['PredictedAnaf'].astype('string').str.len()==3]['ModelId_PredictedAnaf'].unique()

array(['xgb_anaf_e5_v4', 'embedding_anaf_v2', 'bert_anaf_v5'],
      dtype=object)

In [233]:
df_anaf_exp[df_anaf_exp['Predicted_type']=='other, need to check'].head()

,mezaheReshumaRatz,KodSeker,ProjectId,ModelId_PredictedAnaf,PredictedAnaf,Score_PredictedAnaf,CreationDate,Deployed_Enviroment_Version,rank_predict,Predicted_type


In [273]:
df_mishlah_exp.head()

,mezaheReshumaRatz,KodSeker,ProjectId,ModelId_PredictedMishlah,PredictedMishlah,Score_PredictedMishlah,CreationDate,Deployed_Enviroment_Version,rank_predict,Predicted_type
0,112510123177951,12,1,bert_mishlah_v5,8344,0.8876,2026-06-03 13:51:46.217,v0.230,1,valid predict
1,112510123177951,12,1,bert_mishlah_v5,8332,0.9745,2026-06-03 13:51:46.217,v0.230,2,valid predict
2,112510123177951,12,1,bert_mishlah_v5,7511,0.8919,2026-06-03 13:51:46.217,v0.230,3,valid predict
3,112510123177951,12,1,embedding_mishlah_v2,8344,0.9535,2026-06-03 13:51:46.217,v0.230,1,valid predict
4,112510123177951,12,1,embedding_mishlah_v2,0,0.0,2026-06-03 13:51:46.217,v0.230,2,zero


In [272]:
len(df_mishlah_exp[['mezaheReshumaRatz','ModelId_PredictedMishlah']].drop_duplicates())/4

90120.0

In [235]:
group_rank_and_type(df_mishlah_exp, 'Mishlah')

check number of prediction per valid type:


,rank_predict,Predicted_type,cnt,ratio
0,1,valid predict,360480,1.000000
1,2,valid predict,321323,0.891375
2,2,zero,39157,0.108625
3,3,null type,2068,0.005737
4,3,valid predict,298360,0.827674
5,3,zero,60052,0.166589


check number of prediction per model:


,rank_predict,ModelId_PredictedMishlah,cnt_valid,agg_score,ratio_valid
0,1,ReRanker_Mishlah_v3,90120,0.767533,1.000000
1,1,bert_mishlah_v5,90120,0.713266,1.000000
2,1,embedding_mishlah_v2,90120,0.751987,1.000000
3,1,xgb_mishlah_e5_v4,90120,0.759899,1.000000
4,2,ReRanker_Mishlah_v3,90120,0.640691,1.000000
5,2,bert_mishlah_v5,90120,0.646163,1.000000
6,2,embedding_mishlah_v2,50963,0.384054,0.565502
7,2,xgb_mishlah_e5_v4,90120,0.644851,1.000000
8,3,ReRanker_Mishlah_v3,88052,0.555022,0.977053
9,3,bert_mishlah_v5,90120,0.623524,1.000000


In [168]:
df_anaf_exp.head(2)

,mezaheReshumaRatz,KodSeker,ProjectId,ModelId_PredictedAnaf,PredictedAnaf,Score_PredictedAnaf,CreationDate,Deployed_Enviroment_Version,rank_predict,Predicted_type
0,11511123508531,1,1,bert_anaf_v5,85XX,0.8644,2026-07-06 18:08:03.090,v0.234,1,correct predict
1,11511123508531,1,1,bert_anaf_v5,8XXX,0.2595,2026-07-06 18:08:03.090,v0.234,2,correct predict


In [236]:
if (len(df_anaf_exp[df_anaf_exp['Predicted_type']=='other, need to check'])>0) | (len(df_mishlah_exp[df_mishlah_exp['Predicted_type']=='other, need to check'])>0):
    raise  RuntimeError("weird value in predictions")

### test accuracy

#### accuracy results

In [244]:
df_result.head(1)

,Id_Row,mezaheReshumaRatz,KodSeker,ProjectId,PredictedAnaf1,Score_PredictedAnaf1,PredictedMishlah1,Score_PredictedMishlah1,PredictedAnaf2,Score_PredictedAnaf2,PredictedMishlah2,Score_PredictedMishlah2,SentDate,RetrievalDate,StatusTorML,Deployed_Enviroment_Version
0,247489,126158074857211,2,1,None,NaN,2359,1.833,None,NaN,2356,1.5,2026-07-06 14:50:22.210,None,1,v0.230


In [346]:
def get_accuracy_for_results(df_res, df_test, target_name):
    # results and test with actual results
    join_res_test=df_res[['mezaheReshumaRatz',f'Predicted{target_name}1',
              f'Score_Predicted{target_name}1', f'Predicted{target_name}2',
              f'Score_Predicted{target_name}2','StatusTorML']].merge(df_test[['mezaheReshumaRatz',f'Semel{target_name}Sofi']], on='mezaheReshumaRatz')
    
    print(f"number of rows in join results and test is {len(join_res_test)} ")
    print( f"number of rows without  None values in Semel{target_name}Sofi is {len(join_res_test[~join_res_test[f'Semel{target_name}Sofi'].isnull()])}")
    display(join_res_test.head(2))

    print( f"""number of rows with empty code is  {len(join_res_test[(join_res_test[f'Semel{target_name}Sofi']=='') |
                                                                   (join_res_test[f'Predicted{target_name}1']=='')])}""")

    mask=(
        (join_res_test[f'Semel{target_name}Sofi'].notna()) &
        (join_res_test[f'Predicted{target_name}1'].notna()) &
        (join_res_test[f'Semel{target_name}Sofi']!='')

    )
    report_metrics=classification_report(join_res_test[mask][f'Semel{target_name}Sofi'],
                              join_res_test[mask][f'Predicted{target_name}1'],
                              output_dict=True,
                              zero_division=0)
    report_metrics_df=pd.DataFrame(report_metrics).T
    report_metrics_df=report_metrics_df.reset_index().rename(columns={'index':'code'})

    cols_general=['accuracy','macro avg','weighted avg']
    display(report_metrics_df[report_metrics_df['code'].isin(cols_general)])
    
    f1_all=report_metrics_df.loc[report_metrics_df['code']=='weighted avg','f1-score'].iloc[0]
    
    print(f"f1 is {round(f1_all,3)}")
    if f1_all<0.7:
        warnings.warn(f"f1 is less than 70% , the result is {f1_all}")

    return report_metrics_df, join_res_test 


In [347]:
report_metrics_anaf, join_anaf_res_test =get_accuracy_for_results(df_result, df_test, "Anaf")

number of rows in join results and test is 44302 
number of rows without  None values in SemelAnafSofi is 23112


,mezaheReshumaRatz,PredictedAnaf1,Score_PredictedAnaf1,PredictedAnaf2,Score_PredictedAnaf2,StatusTorML,SemelAnafSofi
0,126158074857211,None,NaN,None,NaN,1,None
1,126158074857212,None,NaN,None,NaN,2,None


number of rows with empty code is  8


,code,precision,recall,f1-score,support
545,accuracy,0.810890,0.810890,0.810890,0.81089
546,macro avg,0.570103,0.570339,0.550083,23103.00000
547,weighted avg,0.801973,0.810890,0.797936,23103.00000


f1 is 0.798


In [348]:
report_metrics_mishlah, join_mishlah_res_test =get_accuracy_for_results(df_result, df_test, "Mishlah")

number of rows in join results and test is 44302 
number of rows without  None values in SemelMishlahSofi is 44302


,mezaheReshumaRatz,PredictedMishlah1,Score_PredictedMishlah1,PredictedMishlah2,Score_PredictedMishlah2,StatusTorML,SemelMishlahSofi
0,126158074857211,2359,1.833,2356,1.5,1,2356
1,126158074857212,2359,2.000,1345,1.5,2,1345


number of rows with empty code is  27


,code,precision,recall,f1-score,support
477,accuracy,0.715551,0.715551,0.715551,0.715551
478,macro avg,0.536086,0.523368,0.504239,44275.000000
479,weighted avg,0.715312,0.715551,0.699014,44275.000000


f1 is 0.699


/tmp/ipykernel_1029964/656744533.py:34: UserWarning: f1 is less than 70% , the result is 0.6990137924758884
  warnings.warn(f"f1 is less than 70% , the result is {f1_all}")


#### mishlah and anaf accuracy

In [350]:
df_anaf_exp.head(2)

,mezaheReshumaRatz,KodSeker,ProjectId,ModelId_PredictedAnaf,PredictedAnaf,Score_PredictedAnaf,CreationDate,Deployed_Enviroment_Version,rank_predict,Predicted_type
0,11511123508531,1,1,bert_anaf_v5,85XX,0.8644,2026-07-06 18:08:03.090,v0.234,1,valid predict
1,11511123508531,1,1,bert_anaf_v5,8XXX,0.2595,2026-07-06 18:08:03.090,v0.234,2,valid predict


In [370]:
def get_accuracy_for_anaf_mishlah(df_model, df_test, target_name):
    # results and test with actual results
    join_model_test=df_model[['mezaheReshumaRatz', f'ModelId_Predicted{target_name}','rank_predict',f'Predicted{target_name}',
              f'Score_Predicted{target_name}' ]].merge(df_test[['mezaheReshumaRatz',f'Semel{target_name}Sofi']], on='mezaheReshumaRatz')
    
    print(f"number of rows in join models and test is {len(join_model_test)} ")
    print( f"""number of rows without  None values in Semel{target_name}Sofi is
           {len(join_model_test[~join_model_test[f'Semel{target_name}Sofi'].isnull()])}""")
    display(join_model_test.head(2))

    print( f"""number of rows with empty code is  {len(join_model_test[(join_model_test[f'Semel{target_name}Sofi']=='') |
                                                                   (join_model_test[f'Predicted{target_name}']=='')])}""")

    mask=(
        (join_model_test[f'Semel{target_name}Sofi'].notna()) &
        (join_model_test[f'Predicted{target_name}'].notna()) &
        (join_model_test[f'Semel{target_name}Sofi']!='')

    )
    models= join_model_test[f"ModelId_Predicted{target_name}"].unique().tolist()
    ranking= join_model_test["rank_predict"].unique().tolist()
    dfs=[]
    for m in models:
        for r in ranking:
            print(f"model is {m} and ranking is {r}")
            join_model_test_m=join_model_test[(join_model_test[f'ModelId_Predicted{target_name}']==m) &
                                              (join_model_test['rank_predict']==r)]
            report_metrics=classification_report(join_model_test_m[mask][f'Semel{target_name}Sofi'],
                                    join_model_test_m[mask][f'Predicted{target_name}'],
                                    output_dict=True,
                                    zero_division=0)
            report_metrics_df=pd.DataFrame(report_metrics).T
            report_metrics_df=report_metrics_df.reset_index().rename(columns={'index':'code'})

            cols_general=['accuracy','macro avg','weighted avg']
            display(report_metrics_df[report_metrics_df['code'].isin(cols_general)])
            
            f1_all=report_metrics_df.loc[report_metrics_df['code']=='weighted avg','f1-score'].iloc[0]
            
            print(f"f1 is {round(f1_all,3)}")
            if f1_all<0.7:
                warnings.warn(f"f1 is less than 70% , the result is {f1_all}")
            report_metrics_df['model_name']=m
            report_metrics_df['ranking']=r
            dfs.append(report_metrics_df)

    combined_metric_df=pd.concat(dfs, ignore_index=True)

    return combined_metric_df, join_model_test 

In [362]:
combined_metric_df_anaf, join_model_anaf_test = get_accuracy_for_anaf_mishlah(df_anaf_exp, df_test, "Anaf")

number of rows in join models and test is 277344 
number of rows without  None values in SemelAnafSofi is
           277344


,mezaheReshumaRatz,ModelId_PredictedAnaf,rank_predict,PredictedAnaf,Score_PredictedAnaf,SemelAnafSofi
0,11511123508531,bert_anaf_v5,1,85XX,0.8644,85XX
1,11511123508531,bert_anaf_v5,2,8XXX,0.2595,85XX


number of rows with empty code is  96
model is bert_anaf_v5 and ranking is 1


/tmp/ipykernel_1029964/273946811.py:28: UserWarning: Boolean Series key will be reindexed to match DataFrame index.
  report_metrics=classification_report(join_model_test_m[mask][f'Semel{target_name}Sofi'],
/tmp/ipykernel_1029964/273946811.py:29: UserWarning: Boolean Series key will be reindexed to match DataFrame index.
  join_model_test_m[mask][f'Predicted{target_name}'],


,code,precision,recall,f1-score,support
574,accuracy,0.705938,0.705938,0.705938,0.705938
575,macro avg,0.514230,0.546388,0.504246,23104.000000
576,weighted avg,0.757724,0.705938,0.715891,23104.000000


...

f1 is 0.091
model is xgb_anaf_e5_v4 and ranking is 3


/tmp/ipykernel_1029964/273946811.py:42: UserWarning: f1 is less than 70% , the result is 0.09061575645830806
  warnings.warn(f"f1 is less than 70% , the result is {f1_all}")
/tmp/ipykernel_1029964/273946811.py:28: UserWarning: Boolean Series key will be reindexed to match DataFrame index.
  report_metrics=classification_report(join_model_test_m[mask][f'Semel{target_name}Sofi'],
/tmp/ipykernel_1029964/273946811.py:29: UserWarning: Boolean Series key will be reindexed to match DataFrame index.
  join_model_test_m[mask][f'Predicted{target_name}'],


,code,precision,recall,f1-score,support
627,accuracy,0.028134,0.028134,0.028134,0.028134
628,macro avg,0.030826,0.041931,0.028687,23104.000000
629,weighted avg,0.043097,0.028134,0.027455,23104.000000


f1 is 0.027


/tmp/ipykernel_1029964/273946811.py:42: UserWarning: f1 is less than 70% , the result is 0.02745496363116586
  warnings.warn(f"f1 is less than 70% , the result is {f1_all}")


In [367]:
columns_present=['model_name','ranking','code','precision','recall','f1-score','support']

In [368]:
combined_metric_df_anaf[(combined_metric_df_anaf['code']=="weighted avg") &
                        (combined_metric_df_anaf['ranking']==1)][columns_present]

,model_name,ranking,code,precision,recall,f1-score,support
576,bert_anaf_v5,1,weighted avg,0.757724,0.705938,0.715891,23104.0
2401,embedding_anaf_v2,1,weighted avg,0.673670,0.642529,0.652083,23104.0
4207,ReRanker_Anaf_v3,1,weighted avg,0.796376,0.788954,0.787080,23104.0
6241,xgb_anaf_e5_v4,1,weighted avg,0.814504,0.810163,0.808557,23104.0


In [371]:
combined_metric_df_mishlah, join_model_mishlah_test = get_accuracy_for_anaf_mishlah(df_mishlah_exp, df_test, "Mishlah")

number of rows in join models and test is 531624 
number of rows without  None values in SemelMishlahSofi is
           531624


,mezaheReshumaRatz,ModelId_PredictedMishlah,rank_predict,PredictedMishlah,Score_PredictedMishlah,SemelMishlahSofi
0,112512123485341,bert_mishlah_v5,1,2529,0.6852,2529
1,112512123485341,bert_mishlah_v5,2,2356,0.4186,2529


number of rows with empty code is  324
model is bert_mishlah_v5 and ranking is 1


/tmp/ipykernel_1029964/2964404956.py:28: UserWarning: Boolean Series key will be reindexed to match DataFrame index.
  report_metrics=classification_report(join_model_test_m[mask][f'Semel{target_name}Sofi'],
/tmp/ipykernel_1029964/2964404956.py:29: UserWarning: Boolean Series key will be reindexed to match DataFrame index.
  join_model_test_m[mask][f'Predicted{target_name}'],


,code,precision,recall,f1-score,support
479,accuracy,0.421389,0.421389,0.421389,0.421389
480,macro avg,0.467538,0.398221,0.374134,44275.000000
481,weighted avg,0.631691,0.421389,0.439074,44275.000000


...

f1 is 0.121
model is xgb_mishlah_e5_v4 and ranking is 3


/tmp/ipykernel_1029964/2964404956.py:42: UserWarning: f1 is less than 70% , the result is 0.12072151748167831
  warnings.warn(f"f1 is less than 70% , the result is {f1_all}")
/tmp/ipykernel_1029964/2964404956.py:28: UserWarning: Boolean Series key will be reindexed to match DataFrame index.
  report_metrics=classification_report(join_model_test_m[mask][f'Semel{target_name}Sofi'],
/tmp/ipykernel_1029964/2964404956.py:29: UserWarning: Boolean Series key will be reindexed to match DataFrame index.
  join_model_test_m[mask][f'Predicted{target_name}'],


,code,precision,recall,f1-score,support
571,accuracy,0.047521,0.047521,0.047521,0.047521
572,macro avg,0.033038,0.047616,0.032774,44275.000000
573,weighted avg,0.062951,0.047521,0.049532,44275.000000


f1 is 0.05


/tmp/ipykernel_1029964/2964404956.py:42: UserWarning: f1 is less than 70% , the result is 0.049532181090224225
  warnings.warn(f"f1 is less than 70% , the result is {f1_all}")


In [372]:
combined_metric_df_mishlah[(combined_metric_df_mishlah['code']=="weighted avg") &
                        (combined_metric_df_mishlah['ranking']==1)][columns_present]

,model_name,ranking,code,precision,recall,f1-score,support
481,bert_mishlah_v5,1,weighted avg,0.631691,0.421389,0.439074,44275.0
2005,embedding_mishlah_v2,1,weighted avg,0.623959,0.583399,0.596356,44275.0
3603,ReRanker_Mishlah_v3,1,weighted avg,0.695274,0.655539,0.662656,44275.0
5340,xgb_mishlah_e5_v4,1,weighted avg,0.746767,0.722846,0.728324,44275.0


In [375]:
combined_metric_df_mishlah[(combined_metric_df_mishlah['model_name']=='ReRanker_Mishlah_v3') &
                           (combined_metric_df_mishlah['ranking']==1) ].sort_values('support', ascending=False)

,code,precision,recall,f1-score,support,model_name,ranking
3603,weighted avg,0.695274,0.655539,0.662656,44275.0,ReRanker_Mishlah_v3,1
3602,macro avg,0.494531,0.516572,0.483762,44275.0,ReRanker_Mishlah_v3,1
3600,XXXX,0.677900,0.593307,0.632789,3526.0,ReRanker_Mishlah_v3,1
3208,2512,0.683128,0.625471,0.653029,1327.0,ReRanker_Mishlah_v3,1
3306,3341,0.509317,0.576788,0.540957,853.0,ReRanker_Mishlah_v3,1
...,...,...,...,...,...,...,...
3509,8110,0.000000,0.000000,0.000000,0.0,ReRanker_Mishlah_v3,1
3356,42XX,0.000000,0.000000,0.000000,0.0,ReRanker_Mishlah_v3,1
3521,8133,0.000000,0.000000,0.000000,0.0,ReRanker_Mishlah_v3,1
3536,815X,0.000000,0.000000,0.000000,0.0,ReRanker_Mishlah_v3,1


In [376]:
join_model_mishlah_test.head(1)

,mezaheReshumaRatz,ModelId_PredictedMishlah,rank_predict,PredictedMishlah,Score_PredictedMishlah,SemelMishlahSofi
0,112512123485341,bert_mishlah_v5,1,2529,0.6852,2529


In [384]:
join_model_mishlah_test[(join_model_mishlah_test['ModelId_PredictedMishlah']=="ReRanker_Mishlah_v3") &
                        (join_model_mishlah_test['PredictedMishlah']=="XXXX") &
                         (join_model_mishlah_test['rank_predict']==1) ]

,mezaheReshumaRatz,ModelId_PredictedMishlah,rank_predict,PredictedMishlah,Score_PredictedMishlah,SemelMishlahSofi
54,112512123486761,ReRanker_Mishlah_v3,1,XXXX,0.88353,XXXX
66,112512123487291,ReRanker_Mishlah_v3,1,XXXX,0.88353,XXXX
78,112512123487831,ReRanker_Mishlah_v3,1,XXXX,0.88353,XXXX
114,112512123491011,ReRanker_Mishlah_v3,1,XXXX,0.88353,XXXX
138,112512123491731,ReRanker_Mishlah_v3,1,XXXX,0.88353,XXXX
...,...,...,...,...,...,...
530442,23612607779211,ReRanker_Mishlah_v3,1,XXXX,0.88353,XXXX
530502,23612607815411,ReRanker_Mishlah_v3,1,XXXX,0.88353,XXXX
530958,23612609201511,ReRanker_Mishlah_v3,1,XXXX,0.88353,XXXX
531138,23612609676811,ReRanker_Mishlah_v3,1,XXXX,0.88353,3XXX


In [383]:
join_model_mishlah_test[(join_model_mishlah_test['ModelId_PredictedMishlah']=="ReRanker_Mishlah_v3") &
                        (join_model_mishlah_test['PredictedMishlah']=="XXXX") &
                         (join_model_mishlah_test['rank_predict']==1) ].groupby('SemelMishlahSofi')['mezaheReshumaRatz'].count().reset_index()\
                            .sort_values('mezaheReshumaRatz', ascending=False)

,SemelMishlahSofi,mezaheReshumaRatz
176,XXXX,2092
71,2XXX,90
92,3341,54
84,3321,44
54,2511,34
...,...,...
163,8332,1
161,8321,1
162,832X,1
169,9312,1


In [385]:
join_model_mishlah_test[(join_model_mishlah_test['ModelId_PredictedMishlah']=="ReRanker_Mishlah_v3") &
                        (join_model_mishlah_test['SemelMishlahSofi']=="XXXX") &
                         (join_model_mishlah_test['rank_predict']==1) ]

,mezaheReshumaRatz,ModelId_PredictedMishlah,rank_predict,PredictedMishlah,Score_PredictedMishlah,SemelMishlahSofi
42,112512123486751,ReRanker_Mishlah_v3,1,9629,0.662921,XXXX
54,112512123486761,ReRanker_Mishlah_v3,1,XXXX,0.88353,XXXX
66,112512123487291,ReRanker_Mishlah_v3,1,XXXX,0.88353,XXXX
78,112512123487831,ReRanker_Mishlah_v3,1,XXXX,0.88353,XXXX
114,112512123491011,ReRanker_Mishlah_v3,1,XXXX,0.88353,XXXX
...,...,...,...,...,...,...
530586,23612607973111,ReRanker_Mishlah_v3,1,4211,0.894309,XXXX
530598,23612607979811,ReRanker_Mishlah_v3,1,2512,0.816504,XXXX
530958,23612609201511,ReRanker_Mishlah_v3,1,XXXX,0.88353,XXXX
531426,23612610804311,ReRanker_Mishlah_v3,1,25XX,0.634615,XXXX


In [386]:
join_model_mishlah_test[(join_model_mishlah_test['ModelId_PredictedMishlah']=="ReRanker_Mishlah_v3") &
                        (join_model_mishlah_test['SemelMishlahSofi']=="XXXX") &
                         (join_model_mishlah_test['rank_predict']==1) ].groupby('PredictedMishlah')['mezaheReshumaRatz'].count().reset_index()\
                            .sort_values('mezaheReshumaRatz', ascending=False)

,PredictedMishlah,mezaheReshumaRatz
259,XXXX,2092
229,8212,47
99,3322,43
141,4416,42
108,3341,39
...,...,...
215,8114,1
217,8123,1
220,8135,1
224,8156,1


In [387]:
2092/3526 

0.5933068633011912